[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/llm_ops_overview.ipynb)


<center>
    <p style="text-align:center">
        <img alt="Phoenix 标志" src="https://storage.googleapis.com/arize-phoenix-assets/assets/phoenix-logo-light.svg" width="200"/>
        <br>
        <a href="https://arize.com/docs/phoenix/">文档</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-3r07iavnk-ammtATWSlF0pSrd1DsMW7g">社区</a>
    </p>
</center>
<h1 align="center">Phoenix Python 教程：追踪、评估与实验</h1>

本教程使用 Agno 框架和 OpenAI 模型构建简单的旅行智能体。
1. 首先安装所需的 OpenInference 包，并配置 Arize Phoenix 追踪。

2. 接着配置智能体数据集并上传至 Phoenix，使用 Phoenix Evals 库定义基于 LLM 的评估器，运行实验衡量智能体表现。

3. 然后查看结果、迭代智能体、重新运行实验，并观察改进。

这个端到端教程展示如何通过 Phoenix 追踪、评估并系统地改进应用和智能体。

### **需要免费的 Arize Phoenix Cloud 账号、免费的 Tavily API 密钥，以及 OpenAI API 密钥。**

# 安装依赖并设置密钥

In [ ]:
!pip install -qqqqq arize-phoenix openai agno openinference-instrumentation-openai openinference-instrumentation-agno httpx

In [ ]:
import os
from getpass import getpass

os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = globals().get("PHOENIX_COLLECTOR_ENDPOINT") or getpass(
    "🔑 请输入 Phoenix 端点： "
)

os.environ["PHOENIX_API_KEY"] = globals().get("PHOENIX_API_KEY") or getpass(
    "🔑 请输入 Phoenix API 密钥： "
)

os.environ["OPENAI_API_KEY"] = globals().get("OPENAI_API_KEY") or getpass(
    "🔑 请输入 OpenAI API 密钥： "
)

os.environ["TAVILY_API_KEY"] = globals().get("TAVILY_API_KEY") or getpass(
    "🔑 请输入 Tavily API 密钥： "
)

`phoenix.otel` 的 `register` 函数配置埋点，使智能体自动将追踪发送到 Phoenix。

In [ ]:
from phoenix.otel import register

tracer_provider = register(auto_instrument=True, project_name="python-phoenix-tutorial")

获取 `tracer` 对象，稍后用于手动为部分函数添加埋点。

In [ ]:
from opentelemetry import trace

tracer = trace.get_tracer(__name__)

# 定义智能体

本节构建旅行智能体。用户描述目的地、旅行日期和兴趣后，智能体将生成兼顾预算的个性化行程。

## 定义工具

首先定义支持智能体工具的辅助函数。使用 **Tavily Search** 收集各目的地的通用信息，并通过 **Open-Mateo** 获取天气。

In [ ]:
# --- 工具辅助函数 ---
import httpx


@tracer.chain(name="search-api")
def _search_api(query: str) -> str | None:
    """优先尝试 Tavily 搜索，失败时返回 None。"""
    api_key = os.getenv("TAVILY_API_KEY")

    resp = httpx.post(
        "https://api.tavily.com/search",
        json={
            "api_key": api_key,
            "query": query,
            "max_results": 3,
            "search_depth": "basic",
            "include_answer": True,
        },
        timeout=8,
    )
    data = resp.json()
    answer = data.get("answer", "") or ""
    snippets = [item.get("content", "") for item in data.get("results", [])]

    combined = " ".join([answer] + snippets).strip()
    return combined[:400] if combined else None


@tracer.chain(name="weather-api")
def _weather(dest):
    g = httpx.get(f"https://geocoding-api.open-meteo.com/v1/search?name={dest}")
    if g.status_code != 200 or not g.json().get("results"):
        return ""
    lat, lon = g.json()["results"][0]["latitude"], g.json()["results"][0]["longitude"]
    w = httpx.get(
        f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"
    ).json()
    cw = w.get("current_weather", {})
    return f"当前天气： {cw.get('temperature')}°C, wind {cw.get('windspeed')} km/h."

智能体可以使用三个工具：

1. Essential Info：提供目的地的天气和一般状况等重要旅行信息。
2. Budget Basics：提供旅行成本信息，根据所选活动帮助规划预算。
3. Local Flavor：推荐独特的当地体验和文化亮点。

In [ ]:
from agno.tools import tool


@tool
def essential_info(destination: str) -> str:
    """通过搜索与天气 API 获取必要信息"""
    parts = []

    q = f"{destination} travel essentials weather best time top attractions etiquette"
    s = _search_api(q)
    if s:
        parts.append(f"{destination} essentials: {s}")
    else:
        parts.append(
            f"{destination}是热门旅行目的地，可以探索当地文化、美食与地标。"
        )

    weather = _weather(destination)
    if weather:
        parts.append(weather)

    return f"{destination} essentials:\n" + "\n".join(parts)


@tool
def budget_basics(destination: str, duration: str) -> str:
    """总结旅行费用类别。"""
    q = f"{destination} travel budget average daily costs {duration}"
    s = _search_api(q)
    if s:
        return f"{destination} budget ({duration}): {s}"
    return f"预算： {duration} in {destination}取决于住宿、餐饮、交通与景点。"


@tool
def local_flavor(destination: str, interests: str = "local culture") -> str:
    """推荐地道的当地体验。"""
    q = f"{destination} authentic local experiences {interests}"
    s = _search_api(q)
    if s:
        return f"{destination} {interests}: {s}"
    return f"Explore {destination}'s unique {interests}，通过市场、街区与当地餐馆体验。"

## 构建智能体

接下来构建智能体。Agno 框架支持直接定义模型、指令和工具等关键参数，简化构建过程。

In [ ]:
from agno.agent import Agent
from agno.models.openai import OpenAIChat

# --- 主智能体 ---
trip_agent = Agent(
    name="TripPlanner",
    role="AI Travel Assistant",
    model=OpenAIChat(id="gpt-4o-mini"),
    instructions=(
        "你是友好且知识丰富的旅行规划师。"
        "综合使用多个工具创建旅行计划，包含必要信息、预算与当地特色。"
        "语气自然清晰，控制在 1000 个词以内。"
    ),
    markdown=True,
    tools=[essential_info, budget_basics, local_flavor],
)

In [ ]:
# --- 使用示例 ---
destination = "Tokyo"
duration = "5 days"
interests = "food, culture"

query = f"""
规划持续 {duration}、前往 {destination} 的旅行。
重点关注： {interests}.
包含必要信息、预算明细与当地体验。
"""


def agent_task(query):
    trip_agent.print_response(query, stream=True)


agent_task(query)

# 定义智能体运行的数据集

为智能体开展实验前，先定义运行数据集。这提供了在一致输入下评估智能体行为的标准方式。

本例使用包含十个示例的小数据集，通过 Phoenix Client 上传至 Phoenix。上传后，所有实验都会与该数据集关联追踪。

In [ ]:
from phoenix.client import Client

client = Client()

In [ ]:
import pandas as pd

# --- 示例查询 ---
queries = [
    "Plan a 7-day trip to Italy focused on art, history, and local food. Include essential travel info, a budget estimate, and key attractions in Rome, Florence, and Venice.",
    "Create a 4-day itinerary for Seoul centered on K-pop, fashion districts, and street food. Include transportation tips and a mid-range budget.",
    "Plan a romantic 5-day getaway to Paris with emphasis on museums, wine tasting, and scenic walks. Provide cost estimates and essential travel notes.",
    "Design a 3-day budget trip to Mexico City focusing on food markets, archaeological sites, and nightlife. Include daily cost breakdowns.",
    "Prepare a 6-day itinerary for New Zealand’s South Island with a focus on outdoor adventure, hikes, and photography spots. Include travel logistics and gear essentials.",
    "Plan a 10-day trip across Spain, hitting Barcelona, Madrid, and Seville. 重点关注： architecture, tapas, and cultural festivals. Include a detailed budget.",
    "Create a 5-day family-friendly itinerary for Singapore with theme parks, nature activities, and kid-friendly dining. Include entry fees and transit costs.",
    "Plan a 4-day luxury spa and relaxation trip to Bali. Include premium resorts, wellness activities, and a high-end budget.",
    "Design a 7-day solo backpacking trip through Thailand with hostels, street food, and cultural attractions. Provide safety essentials and budget breakdown.",
    "Create a 3-day weekend itinerary for New York City focusing on art galleries, rooftop restaurants, and iconic attractions. Include estimated costs.",
]

dataset_df = pd.DataFrame(data={"input": queries})

dataset = client.datasets.create_dataset(
    dataframe=dataset_df, name="travel-questions", input_keys=["input"]
)

![上传至 Phoenix 的数据集](https://storage.googleapis.com/arize-phoenix-assets/assets/images/end-to-end-python-tutorial-dataset.png)

# 定义评估器

接下来评估智能体的输出。Evals 提供结构化方法，衡量回答是否满足数据集中定义的准确性、相关性、一致性或安全性等要求。

本教程使用 LLM 裁判评估，由另一个 LLM 充当评估器。先在提示词中定义明确的评估标准，再将每个数据集样本的输入和智能体输出传给评估器。评估 LLM 返回分数及自然语言解释，说明评分原因。

这样可以自动为大量样本评分，既获得量化指标，也深入理解失败案例。

![Phoenix 评估](https://storage.googleapis.com/arize-phoenix-assets/assets/images/phoenix_evals_diagram.png)

In [ ]:
ANSWER_RELEVANCE_PROMPT_TEMPLATE = """
你会收到旅行规划查询与行程答案。请判断答案是否正确遵循用户指令。
如果答案违背、忽略或缺少查询中的必要元素，例如旅行时长、目的地、主题、预算细节或必要信息，
则为 "incorrect"。添加无关或矛盾细节时也为 "incorrect"。

    [BEGIN DATA]
    ************
    [查询]：{{input}}
    ************
    [答案]：{{output}}
    ************
    [END DATA]

逐步解释判断过程，然后给出最终标签：
- 答案准确完整遵循查询时，使用 "correct"。
- 偏离查询或遗漏必要信息时，使用 "incorrect"。

最终输出必须仅为一个词："correct" 或 "incorrect"。
"""

定义评估提示词后，使用 Phoenix Evals 库构建评估器实例。注意，评估器使用的 LLM 与智能体自身使用的模型相互独立。

In [ ]:
from phoenix.evals import ClassificationEvaluator
from phoenix.evals.llm import LLM

llm = LLM(provider="openai", model="gpt-5")

relevancy_evaluator = ClassificationEvaluator(
    name="答案相关性",
    llm=llm,
    prompt_template=ANSWER_RELEVANCE_PROMPT_TEMPLATE,
    choices={"correct": 1.0, "incorrect": 0.0},
)

In [ ]:
BUDGET_CONSISTENCY_PROMPT_TEMPLATE = """
你会收到旅行规划查询与行程答案。请判断答案的预算是否一致且在数学上合理。
存在以下情况时为 "incorrect"：
- 所列全部预算类别的最低费用之和高于给定最低总估算。
- 所列全部预算类别的最高费用之和高于给定最高总估算。
- 总估算范围无法由分项范围推导，或与分项范围矛盾。
- 列出预算项目，但总额与分项数值不一致。
- 价格或费用范围自相矛盾。

    [BEGIN DATA]
    ************
    [查询]：{{input}}
    ************
    [答案]：{{output}}
    ************
    [END DATA]

逐步解释如何评估分项费用和最终总额，包括范围是否在数学上匹配。然后给出最终标签：
- 总预算与分项值一致时，使用 "correct"。
- 总额矛盾或无法由分项值推导时，使用 "incorrect"。

最终输出必须仅为一个词："correct" 或 "incorrect"。
"""

In [ ]:
llm = LLM(provider="openai", model="gpt-5")

budget_evaluator = ClassificationEvaluator(
    name="预算一致性",
    llm=llm,
    prompt_template=BUDGET_CONSISTENCY_PROMPT_TEMPLATE,
    choices={"correct": 1.0, "incorrect": 0.0},
)

# 运行实验

运行实验前的最后一步是显式定义任务。虽然知道要评估智能体，仍需将其包装成一个接收输入并返回智能体输出的简单函数。实验会对数据集中每个示例执行此任务。

In [ ]:
def agent_task(input):
    query = input["input"]
    response = trip_agent.run(query, stream=False)
    return response.content

定义任务后，通过提供数据集、评估器及相关元数据构建实验。配置完成即可运行。

运行时间取决于数据集大小、任务复杂度和评估器数量，可能需要几分钟。

In [ ]:
from phoenix.client.experiments import run_experiment

experiment = run_experiment(
    dataset=dataset,
    task=agent_task,
    experiment_name="inital run",
    evaluators=[relevancy_evaluator, budget_evaluator],
)

# 分析实验追踪与结果

实验已针对每个数据集示例运行，现在可以在 Phoenix 中探索结果：

- 查看智能体产生的完整追踪，逐步检查每次操作
- 检查每个示例的评估输出，包括分数、标签和解释
- 检查评估过程自身的追踪，即 LLM 裁判的推理过程
- 查看整个数据集的汇总评估分数

<video controls width="100%">
  <source src="https://storage.googleapis.com/arize-phoenix-assets/assets/images/end-to-end-python-tutorial-experiment.mp4" type="video/mp4">
  你的浏览器不支持 video 标签。
</video>

### 人工标注

除了 Evals，还可以为追踪添加人工标注，记录明确反馈、标记有问题的输出、突出优秀回答，以及保存自动评估器可能遗漏的观察。人工标注作为追踪的一部分保存，帮助指导应用或智能体的后续迭代。

<video controls width="100%">
  <source src="https://storage.googleapis.com/arize-phoenix-assets/assets/images/end-to-end-python-annotations.mp4" type="video/mp4">
  你的浏览器不支持 video 标签。
</video>

# 迭代并重新运行实验

分析实验结果后，开始迭代。

由于预算计算的评估分数较低，下面修改智能体的主提示词，使其更明确地处理预算计算。也可以修改提示词或做出其他改进，再次运行实验，观察输出如何改善，以及是否出现退步。

迭代是优化智能体行为的关键，每次实验都提供有价值的反馈。当评估分数和输出达到预期时，即可有依据地将变更用于生产。

In [ ]:
# --- 使用更新指令的主智能体 ---
trip_agent = Agent(
    name="TripPlanner",
    role="AI Travel Assistant",
    model=OpenAIChat(id="gpt-4o-mini"),
    instructions=(
        "你是友好且知识丰富的旅行规划师。"
        "综合使用多个工具创建旅行计划，包含必要信息、预算与当地特色。"
        "语气自然清晰，控制在 1000 个词以内。 "
        "提供预算细节时，确保最终总预算范围与各分项范围之和在数学上保持一致。"
    ),
    markdown=True,
    tools=[essential_info, budget_basics, local_flavor],
)

In [ ]:
experiment = run_experiment(
    dataset=dataset,
    task=agent_task,
    experiment_name="updated agent prompt to improve budget",
    evaluators=[relevancy_evaluator, budget_evaluator],
)

本例中，预算一致性分数实际上下降了，而回答相关性提高了。查看追踪和评估解释，可以理解原因：也许更严格的提示词引入了新的边界情况，或智能体以意想不到的方式过度纠正。

接下来可以提出新假设，根据观察调整，再运行实验，持续迭代。每个循环都会优化智能体行为，使输出更稳定地满足目标标准。


![更新后的结果](https://storage.googleapis.com/arize-phoenix-assets/assets/images/end-to-end-python-tutorial-prompt-iteration.png)